In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# MPA-FER reproduction - Colab runner\n",
    "\n",
    "Paper: *Multimodal Prompt Alignment for Facial Expression Recognition* (arXiv 2506.21017).\n",
    "\n",
    "**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.\n",
    "\n",
    "Run the cells top to bottom. Cells 1-5 are setup, cell 6 is the no-dataset\n",
    "sanity check, cells 7-9 are the real short training run."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Check the GPU"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!nvidia-smi\n",
    "import torch\n",
    "print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Get the code\n",
    "\n",
    "Re-running this cell pulls your latest commits, so push from VS Code and\n",
    "re-run rather than editing files inside Colab."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "REPO = 'MPA_FER_PYTORCH_REPRODUCED'\n",
    "URL = 'https://github.com/varsityboi/MPA_FER_PYTORCH_REPRODUCED.git'\n",
    "\n",
    "if not os.path.isdir(f'/content/{REPO}'):\n",
    "    !git clone $URL /content/$REPO\n",
    "%cd /content/$REPO\n",
    "!git pull --ff-only\n",
    "!ls"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Install dependencies (~1 min; torch is already on Colab)"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install -q -r requirements.txt\n",
    "import clip; print('CLIP models available:', clip.available_models())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Save outputs to Google Drive (optional but recommended)\n",
    "\n",
    "Colab wipes `/content` when the session ends. Putting `output.dir` on Drive\n",
    "means checkpoints, logs and the prototype cache survive, and `train.py`\n",
    "resumes from `last.pt` automatically."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "USE_DRIVE = True\n",
    "\n",
    "if USE_DRIVE:\n",
    "    from google.colab import drive\n",
    "    drive.mount('/content/drive')\n",
    "    OUT_DIR = '/content/drive/MyDrive/mpa_fer/outputs/rafdb_vitb16'\n",
    "else:\n",
    "    OUT_DIR = 'outputs/rafdb_vitb16'\n",
    "os.makedirs(OUT_DIR, exist_ok=True)\n",
    "print('outputs ->', OUT_DIR)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Get RAF-DB\n",
    "\n",
    "RAF-DB needs a licence agreement with the authors, so it is not a public\n",
    "download. Pick whichever route you actually have access to:\n",
    "\n",
    "* **A** - a Kaggle mirror, via `kagglehub` (needs your `kaggle.json` token)\n",
    "* **B** - a zip you already put on your Drive\n",
    "\n",
    "`dataset.py` sniffs the folder layout, so either the official\n",
    "`basic/EmoLabel + basic/Image/aligned` structure or `train/1..7` +\n",
    "`test/1..7` class folders will work."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- Option A: Kaggle mirror -------------------------------------------\n",
    "# from google.colab import files; files.upload()        # upload kaggle.json\n",
    "# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json\n",
    "# import kagglehub\n",
    "# DATA_ROOT = kagglehub.dataset_download('<owner>/<rafdb-dataset-slug>')\n",
    "\n",
    "# ---- Option B: zip on your Drive ---------------------------------------\n",
    "# !unzip -q '/content/drive/MyDrive/mpa_fer/rafdb.zip' -d /content/rafdb\n",
    "# DATA_ROOT = '/content/rafdb'\n",
    "\n",
    "DATA_ROOT = '/content/rafdb'   # <- set this to wherever the data ended up\n",
    "print(DATA_ROOT)\n",
    "!ls -R $DATA_ROOT | head -30"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Confirm the loader sees the data before spending GPU time on it\n",
    "from dataset import RAFDB, detect_layout, build_transforms\n",
    "from utils.misc import load_config\n",
    "\n",
    "cfg = load_config('configs/rafdb.yaml', [f'data.root={DATA_ROOT}'])\n",
    "print('layout:', detect_layout(DATA_ROOT))\n",
    "for split in ['train', 'test']:\n",
    "    ds = RAFDB(DATA_ROOT, split, cfg['data']['class_names'],\n",
    "               transform=build_transforms(cfg, train=(split == 'train')))\n",
    "    print(f'{split}: {len(ds)} images, per class {ds.class_counts()}')\n",
    "\n",
    "# RAF-DB should report 12271 train / 3068 test\n",
    "img, label = ds[0]\n",
    "print('tensor', tuple(img.shape), 'label', label, '=', cfg['data']['class_names'][label])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Sanity check (no dataset needed)\n",
    "\n",
    "Fake images, ~3 minutes. Checks frozen CLIP is untouched, gradients reach\n",
    "only the prompts, losses stay finite under fp16, checkpoints round-trip,\n",
    "and prints the per-epoch time estimate. Fix any `FAIL` before moving on."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": ["!python tools/sanity_check.py"]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Build the class prototypes (Eq. 6)\n",
    "\n",
    "One pass of the frozen encoder over the training set. Cached in `OUT_DIR`,\n",
    "so this only runs once."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python tools/build_prototypes.py --set data.root=$DATA_ROOT output.dir=$OUT_DIR"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8a. Two-minute smoke run\n",
    "\n",
    "1 epoch capped at 20 iterations. Only proves the pipeline turns over -\n",
    "accuracy will be garbage."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python train.py --set data.root=$DATA_ROOT output.dir=$OUT_DIR/smoke \\\n",
    "    train.epochs=1 train.max_iters_per_epoch=20 train.log_every=5 output.resume=false"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8b. The real 3-epoch check\n",
    "\n",
    "This is the run you look at: losses should fall every epoch and test\n",
    "accuracy should already be well above chance (14% for 7 classes)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python train.py --set data.root=$DATA_ROOT output.dir=$OUT_DIR train.epochs=3"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 9. Plot the loss curves"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json, matplotlib.pyplot as plt\n",
    "\n",
    "rows = [json.loads(l) for l in open(f'{OUT_DIR}/metrics.jsonl')]\n",
    "epochs = [r['epoch'] for r in rows]\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "for key in ['loss_total', 'loss_vt', 'loss_ta', 'loss_pa', 'loss_v']:\n",
    "    if key in rows[0]:\n",
    "        axes[0].plot(epochs, [r[key] for r in rows], marker='o', label=key)\n",
    "axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].grid(alpha=.3)\n",
    "axes[0].set_title('Training losses')\n",
    "\n",
    "axes[1].plot(epochs, [r['train_acc'] for r in rows], marker='o', label='train')\n",
    "if 'acc' in rows[-1]:\n",
    "    ep = [r['epoch'] for r in rows if 'acc' in r]\n",
    "    axes[1].plot(ep, [r['acc'] for r in rows if 'acc' in r], marker='s', label='test')\n",
    "axes[1].axhline(100 / 7, ls='--', c='gray', label='chance')\n",
    "axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy %'); axes[1].legend(); axes[1].grid(alpha=.3)\n",
    "axes[1].set_title('Accuracy')\n",
    "plt.tight_layout(); plt.show()\n",
    "\n",
    "for r in rows:\n",
    "    print({k: (round(v, 4) if isinstance(v, float) else v)\n",
    "           for k, v in r.items() if k != 'per_class_acc'})"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Ablations (Table 1) - once the 3-epoch run looks sane\n",
    "\n",
    "Each row turns one component off. Run them with the same short schedule so\n",
    "the numbers are comparable to each other, not to the paper."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ablations = {\n",
    "    'coop_baseline': 'model.visual_prompts.enabled=false model.local_align.enabled=false '\n",
    "                     'losses.soft_hard_align.enabled=false losses.proto_align.enabled=false',\n",
    "    'plus_visual':   'model.local_align.enabled=false losses.soft_hard_align.enabled=false '\n",
    "                     'losses.proto_align.enabled=false',\n",
    "    'plus_proto':    'model.local_align.enabled=false losses.soft_hard_align.enabled=false',\n",
    "    'plus_softhard': 'model.local_align.enabled=false',\n",
    "    'full':          '',\n",
    "}\n",
    "\n",
    "for name, flags in ablations.items():\n",
    "    print(f'\\n===== {name} =====')\n",
    "    !python train.py --set data.root=$DATA_ROOT output.dir=$OUT_DIR/abl_{name} \\\n",
    "        train.epochs=3 output.resume=false {flags}"
   ]
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {"provenance": []},
  "kernelspec": {"display_name": "Python 3", "name": "python3"},
  "language_info": {"name": "python"}
 },
 "nbformat": 4,
 "nbformat_minor": 0
}